# 10 - Error Analysis

## Objective

Menganalisis error pada rule deterministic `agreement_count >= 2` menggunakan label manual dari review queue.

## Focus

1. Mengidentifikasi 7 false positive pada rule `agreement_count >= 2`.
2. Membandingkan pair low-agreement dan medium-agreement.
3. Menemukan pola field yang menyebabkan false positive.
4. Mendokumentasikan keterbatasan validation sample.

## Error categories

- `shared_weak_fields`: agreement hanya pada field yang tidak cukup discriminative.
- `blocking_collision`: pair masuk candidate karena blocking, tetapi agreement identity rendah.
- `threshold_too_loose`: pair memiliki agreement minimal dua tetapi label manual berbeda entity.
- `missing_or_unavailable_evidence`: field identity tidak tersedia pada kedua row sehingga evidence terbatas.
- `review_sample_limitation`: temuan berasal dari sample manual, bukan seluruh candidate set.

## Limitations

`review_label` adalah hasil manual review sample. Analisis ini tidak menggunakan `customer_id` sebagai ground truth, tidak mengestimasi error pada seluruh dataset, dan tidak melakukan automatic merge.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_CANDIDATES = [
    Path.cwd() / 'data' / 'processed' / 'crm_50000_customers_standardized.csv',
    Path.cwd().parent / 'data' / 'processed' / 'crm_50000_customers_standardized.csv',
]
DATA_PATH = next((path.resolve() for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Dataset terstandardisasi tidak ditemukan.')

PROCESSED_DIR = DATA_PATH.parent
REVIEW_QUEUE_PATH = PROCESSED_DIR / 'manual_review_queue.csv'
COMPARISON_PATH = PROCESSED_DIR / 'probabilistic_comparison_vectors.csv'
if not REVIEW_QUEUE_PATH.exists():
    raise FileNotFoundError('manual_review_queue.csv belum tersedia.')
if not COMPARISON_PATH.exists():
    raise FileNotFoundError('probabilistic_comparison_vectors.csv belum tersedia.')

review_queue = pd.read_csv(REVIEW_QUEUE_PATH, sep=';')
comparison = pd.read_csv(COMPARISON_PATH)
review_queue['review_label'] = pd.to_numeric(review_queue['review_label'], errors='coerce')
review_queue['agreement_count'] = pd.to_numeric(review_queue['agreement_count'], errors='coerce')

print('Review queue:', review_queue.shape)
print('Comparison vectors:', comparison.shape)
print('Labeled rows:', int(review_queue['review_label'].notna().sum()))

## Experiment 1 - Reconstruct predictions and verify labels

Prediksi error analysis dibuat eksplisit: pair dianggap predicted match jika `agreement_count >= 2`. Label `1` berarti same entity dan label `0` berarti different entity.

In [ ]:
required_columns = {
    'left_row_index', 'right_row_index', 'agreement_count',
    'available_field_count', 'review_label', 'evidence_band',
    'supporting_blocking_strategies',
}
missing_columns = sorted(required_columns - set(review_queue.columns))
if missing_columns:
    raise ValueError(f'Kolom review belum tersedia: {missing_columns}')

reviewed = review_queue.dropna(subset=['review_label']).copy()
reviewed['review_label'] = reviewed['review_label'].astype(int)
reviewed['predicted_match_ge_2'] = reviewed['agreement_count'].ge(2).astype(int)
reviewed['predicted_match_ge_4'] = reviewed['agreement_count'].ge(4).astype(int)

error_counts = pd.DataFrame({
    'error_type': ['true_positive', 'false_positive', 'false_negative', 'true_negative'],
    'count': [
        int(((reviewed['review_label'] == 1) & (reviewed['predicted_match_ge_2'] == 1)).sum()),
        int(((reviewed['review_label'] == 0) & (reviewed['predicted_match_ge_2'] == 1)).sum()),
        int(((reviewed['review_label'] == 1) & (reviewed['predicted_match_ge_2'] == 0)).sum()),
        int(((reviewed['review_label'] == 0) & (reviewed['predicted_match_ge_2'] == 0)).sum()),
    ],
})
error_counts

## Experiment 2 - False positive inventory

Inventory ini berisi row index, provenance blocking, agreement pattern, dan catatan manual. Nilai customer tidak ditampilkan dalam output ringkasan.

In [ ]:
false_positive_columns = [
    'left_row_index', 'right_row_index',
    'supporting_blocking_strategies', 'blocking_strategy_count',
    'max_block_size', 'agreement_pattern', 'agreement_count',
    'available_field_count', 'evidence_band', *[
        'email_agree', 'phone_agree', 'name_agree',
        'address_agree', 'city_agree', 'dob_agree',
    ], 'review_label', 'review_notes',
]
false_positives = reviewed[
    (reviewed['predicted_match_ge_2'] == 1) & (reviewed['review_label'] == 0)
][false_positive_columns].copy()
false_positives['error_category'] = 'threshold_too_loose'

print('False positives:', len(false_positives))
false_positives

## Experiment 3 - Field agreement and blocking provenance

Analisis ini mencari field agreement yang dominan pada false positive. Agreement pada DOB, city, atau phone prefix tidak otomatis berarti same entity karena field tersebut dapat mengalami collision.

In [ ]:
agreement_columns = [
    'email_agree', 'phone_agree', 'name_agree',
    'address_agree', 'city_agree', 'dob_agree',
]
false_positive_agreement = (
    false_positives[agreement_columns]
    .sum()
    .rename('false_positive_count')
    .reset_index()
    .rename(columns={'index': 'agreement_field'})
)

false_positive_provenance = (
    false_positives['supporting_blocking_strategies']
    .value_counts()
    .rename_axis('supporting_blocking_strategies')
    .reset_index(name='false_positive_count')
)

false_positive_agreement, false_positive_provenance

## Experiment 4 - Low and medium agreement analysis

Low agreement didefinisikan sebagai `agreement_count < 2`; medium agreement sebagai `agreement_count` 2 sampai 3. Band ini dipakai untuk error analysis, bukan threshold final.

In [ ]:
reviewed['agreement_band'] = pd.cut(
    reviewed['agreement_count'],
    bins=[-1, 1, 3, 6],
    labels=['low_agreement', 'medium_agreement', 'high_agreement'],
)
agreement_band_summary = (
    reviewed.groupby('agreement_band', observed=False)
    .agg(
        pair_count=('agreement_count', 'size'),
        reviewed_same_entity=('review_label', 'sum'),
        reviewed_different_entity=('review_label', lambda values: int((values == 0).sum())),
        mean_available_field_count=('available_field_count', 'mean'),
    )
    .reset_index()
)
agreement_band_summary['observed_same_entity_rate'] = (
    agreement_band_summary['reviewed_same_entity']
    .div(agreement_band_summary['pair_count'])
)
agreement_band_summary

## Experiment 5 - Root-cause classification

Klasifikasi berikut bersifat rule-based diagnostic. Ia membantu memilih kasus yang perlu ditinjau, bukan menggantikan review manual.

In [ ]:
def classify_root_cause(row: pd.Series) -> str:
    if row['review_label'] == 0 and row['agreement_count'] >= 2:
        if row['agreement_pattern'] in {'000011', '000010'}:
            return 'shared_weak_fields'
        if row['blocking_strategy_count'] == 1:
            return 'blocking_collision'
        return 'threshold_too_loose'
    if row['agreement_count'] < 2:
        return 'insufficient_identity_evidence'
    return 'not_error_in_review_sample'

reviewed['root_cause_category'] = reviewed.apply(classify_root_cause, axis=1)
root_cause_summary = (
    reviewed.groupby('root_cause_category')
    .size()
    .rename('pair_count')
    .reset_index()
    .sort_values('pair_count', ascending=False)
)
root_cause_summary

## Experiment 6 - Sample limitation audit

Metrik sample harus dibaca bersama ukuran candidate set dan cara sampling. Queue ini adalah sample terstratifikasi dari candidate pairs hasil blocking, bukan sample acak dari seluruh N x N pair.

In [ ]:
comparison_pair_count = len(comparison)
reviewed_pair_count = len(reviewed)
limitation_summary = pd.DataFrame({
    'metric': [
        'all_comparison_vector_pairs',
        'reviewed_pairs',
        'reviewed_pair_percentage_of_comparison_vectors',
        'reviewed_same_entity_pairs',
        'reviewed_different_entity_pairs',
        'reviewed_sample_has_ground_truth',
        'candidate_pairs_outside_review_sample_not_evaluated',
    ],
    'value': [
        comparison_pair_count,
        reviewed_pair_count,
        reviewed_pair_count / comparison_pair_count * 100 if comparison_pair_count else 0,
        int((reviewed['review_label'] == 1).sum()),
        int((reviewed['review_label'] == 0).sum()),
        bool(reviewed_pair_count > 0),
        True,
    ],
})
limitation_summary

# Result, Analysis, and Decision

## Result aktual

Gunakan `error_counts`, `false_positives`, `false_positive_agreement`, `false_positive_provenance`, `agreement_band_summary`, `root_cause_summary`, dan `limitation_summary` sebagai sumber hasil aktual. Jangan menulis ulang angka secara manual.

## Interpretation

- False positive pada `agreement_count >= 2` menunjukkan dua agreement belum cukup sebagai automatic merge rule.
- Low agreement pair perlu diperlakukan sebagai candidate untuk review atau fuzzy comparison, bukan match.
- High agreement pada sample ini harus tetap diperlakukan sebagai evidence dari sample, bukan jaminan populasi.
- Jika false positive banyak berasal dari provenance satu blocking key, blocking key tersebut perlu diperketat atau dipakai hanya sebagai candidate generator.

## Decision

Rule `agreement_count >= 4` dapat dipertahankan sebagai high-confidence review candidate pada sample ini, tetapi belum menjadi keputusan merge final. Rule `agreement_count >= 2` tidak aman untuk automatic merge karena menghasilkan false positive.

## Next Experiment

Gunakan hasil error analysis untuk `11_benchmark.ipynb`: bandingkan runtime, candidate count, memory, dan metrik rule deterministic/fuzzy pada validation sample yang sama. Tambahkan manual labels baru bila ingin menguji generalisasi di luar 107 pair.

In [ ]:
OUTPUT_DIR = DATA_PATH.parent
ERROR_COUNTS_PATH = OUTPUT_DIR / 'error_counts.csv'
FALSE_POSITIVE_PATH = OUTPUT_DIR / 'false_positive_analysis.csv'
AGREEMENT_PATH = OUTPUT_DIR / 'false_positive_agreement_summary.csv'
PROVENANCE_PATH = OUTPUT_DIR / 'false_positive_provenance_summary.csv'
BAND_PATH = OUTPUT_DIR / 'agreement_band_summary.csv'
ROOT_CAUSE_PATH = OUTPUT_DIR / 'root_cause_summary.csv'
LIMITATION_PATH = OUTPUT_DIR / 'error_analysis_limitations.csv'

error_counts.to_csv(ERROR_COUNTS_PATH, index=False)
false_positives.to_csv(FALSE_POSITIVE_PATH, index=False)
false_positive_agreement.to_csv(AGREEMENT_PATH, index=False)
false_positive_provenance.to_csv(PROVENANCE_PATH, index=False)
agreement_band_summary.to_csv(BAND_PATH, index=False)
root_cause_summary.to_csv(ROOT_CAUSE_PATH, index=False)
limitation_summary.to_csv(LIMITATION_PATH, index=False)

print('Saved error-analysis artifacts to:', OUTPUT_DIR)
print('Raw dataset still exists:', (DATA_PATH.parents[1] / 'raw' / 'crm_50000_customers_dirty_v3.csv').exists())